# SCRESIA — Tail-resilience: PPO vs best static (review + experiment)

**For Thom & David.** Two goals: (1) read whether RL beats a well-chosen static on the
*right* resilience metric, and (2) let David run his own experiments and propose how to
make the model converge.

## What we changed in the environment (faithfulness fixes)
We made the env faithful to Garrido (2017) *before* judging RL, because earlier 'wins'
were artifacts of a broken world:
- **`risk_occurrence_mode=thesis_periodic`** — fixes a ~2x risk-frequency bug; now matches
  Table 6.11 (PASS). (Old `legacy_renewal` drew inter-arrivals with mean ~b/2 -> 2x events.)
- **`raw_material_flow_mode=kit_equivalent_order_up_to`** — the It,S inventory buffer used
  to be inert (open-loop supply drowned it ~37x). Now it is order-up-to, so the buffer
  actually binds and moderates ReT under disruption (reproduces Garrido H2, under severe).
- **`stochastic_pt`** — right-skewed Tri(0.75,1.0,1.5)xPT processing times.
- Action contract **`thesis_factorized = MultiDiscrete([6,3])`** = Garrido's EXACT two
  decision variables: one inventory level `I_{t,S}` + one shift level `S`.

## Why the MEAN ReT lies (the saturation problem)
Even random/bad policies look near-perfect on the mean: 76-91% of orders are in the
*normal* 'fill-rate case' (ReT approx 1), so the mean is approx 1. **Resilience lives in the
tail / disruption windows.** Judge victory there, not on the mean or the training reward.

## Correct metrics (Codex, commit `d2c5703`)
| role | column | note |
|---|---|---|
| PRIMARY all-order ReT | `ret_mean_all_orders_zero_unfulfilled_mean` | unfulfilled count as 0 |
| tail ReT | `ret_p10_all_mean`, `ret_p50_all_mean` | the discriminating signal |
| service under stress | `flow_fill_rate_mean`, `stockout_week_pct_mean` | |
| saturation indicator | `pct_case_fill_rate_mean` | % orders never disrupted |
| decomposition | `re_{fr,ap,rp,dp_rp}_contribution_all_mean` | fill/autotomy/recovery/non-rec |
| AVOID | `order_level_ret_mean` (legacy) | averages only COMPLETED orders -> optimistic |


In [ ]:
# ============================================================
#  EXPERIMENT KNOBS  --  David: edit this cell and Run All
# ============================================================

# --- model / learning ---
ALGO = 'ppo_mlp'           # 'ppo_mlp' | 'recurrent_ppo' | 'dmlpa_ppo'
                           #   recurrent_ppo/dmlpa_ppo test David's hypothesis that an
                           #   MLP is too weak to ANTICIPATE disruptions (needs memory).
HISTORY_WINDOW = 30        # only used by dmlpa_ppo (transformer over obs history)
POLICY_NET_ARCH = 'medium' # 'small' | 'medium' | 'large'
LEARNING_RATE = 3e-4
TRAIN_TIMESTEPS = 100_000
EVAL_EPISODES = 20
N_ENVS = 1                 # parallel training envs. 1 = prior behavior; 8 = lower-variance
                           #   gradients (helps convergence, esp. recurrent_ppo). Per-episode
                           #   disruptions still re-randomize, so this is not seed-overfitting.

# --- reward (ReT_ladder_v1 = the steep, recovery-weighted reward) ---
REWARD_MODE = 'ReT_ladder_v1'   # also: 'ReT_seq_v1', 'control_v1', 'ReT_cd_v1'
LADDER_W_SC, LADDER_W_RC, LADDER_W_EF = 0.65, 0.30, 0.05   # service / recovery / efficiency
NORM_REWARD = True         # VecNormalize norm_reward (David's ask). Scale-only transform:
                           #   helps value-fit/convergence, does NOT change optimal policy or
                           #   bias the RL-vs-static comparison. obs-norm is already ON by default.

# --- world / stress ---
RISK_LEVELS = ['increased', 'severe']   # dose-response: mean-saturated(tail alive) vs harsh
SEEDS = [42, 303]
TRAIN_CFIS = ''            # '' = fixed risk-level env; '31-90' = train across the Cf panel
                           #   (panel training is the honest setup for generalization)

# --- convergence diagnostic (optional, multiplies cost) ---
# If non-empty, retrains at each budget for ONE (risk, seed) and plots the eval metric
# vs budget so David can SEE whether the policy is learning or stuck.
CONVERGENCE_BUDGETS = []   # e.g. [25_000, 50_000, 100_000]

# --- fixed faithfulness contract (do not weaken without a declared reason) ---
ACTION_SPACE_MODE = 'thesis_factorized'        # Garrido's exact [6,3]
RISK_OCCURRENCE_MODE = 'thesis_periodic'       # Table 6.11 PASS
RAW_MATERIAL_FLOW_MODE = 'kit_equivalent_order_up_to'
ORDER_UP_TO_MULTIPLIER = 1.0   # stress-extension dial, NOT the thesis-faithful backbone.
                               #   m2.0 reproduces Table 6.10 best (~738k/year, -3.8% vs ECS)
                               #   and is the Paper-1 thesis-faithful setting. m1.0 is a declared
                               #   raw-material scarcity/headroom scout; do not mix it into claims
                               #   about strict Garrido replication.
STOCHASTIC_PT = True
GARRIDO_CFIS = '31'        # minimal Garrido baseline (not the focus here)

GIT_URL = 'https://github.com/Thom-320/scres-ia.git'
GIT_BRANCH = 'codex/garrido-postfix-reruns'   # tail wiring lives ONLY on this branch
REPO_NAME = 'scresia'
RUNTIME_PIP_PACKAGES = ['simpy>=4.1', 'numpy>=1.26', 'gymnasium>=0.29',
    'stable-baselines3>=2.3', 'sb3-contrib>=2.3', 'pandas>=2.2', 'scipy>=1.10', 'matplotlib>=3.7']


In [ ]:
# ---- portable Kaggle/Colab/local setup ----
import os, sys, subprocess, json, shutil
from pathlib import Path
from datetime import datetime, timezone

IN_KAGGLE = Path('/kaggle/working').exists()
IN_COLAB = 'google.colab' in sys.modules

def run_cmd(cmd, cwd=None, check=True):
    print('$', ' '.join(map(str, cmd)), flush=True)
    r = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-8000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f'cmd failed rc={r.returncode}: {cmd}')
    return r

if IN_KAGGLE or IN_COLAB:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', *RUNTIME_PIP_PACKAGES])

base_dir = Path('/kaggle/working') if IN_KAGGLE else Path('/content') if IN_COLAB else Path.cwd()
repo_root = base_dir / REPO_NAME
if not repo_root.exists():
    run_cmd(['git', 'clone', '--depth', '1', '--branch', GIT_BRANCH, GIT_URL, str(repo_root)])
else:
    print('repo already present:', repo_root)
output_root = base_dir / 'tail_resilience'
output_root.mkdir(parents=True, exist_ok=True)
print('branch:', run_cmd(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=repo_root).stdout.strip())
print('head:', run_cmd(['git', 'rev-parse', 'HEAD'], cwd=repo_root).stdout.strip())


In [ ]:
# ---- verify the branch has the tail wiring + faithful modes (fail early otherwise) ----
smoke_src = (repo_root / 'scripts/run_thesis_decision_ppo_smoke.py').read_text()
assert 'ret_mean_all_orders_zero_unfulfilled_mean' in smoke_src, 'no tail wiring; wrong branch'
assert 'order_metric_distribution' in smoke_src, 'no order_metric_distribution import'
assert '--norm-reward' in smoke_src, 'no reward-normalization flag; wrong branch'
cfg_src = (repo_root / 'supply_chain/config.py').read_text()
assert 'kit_equivalent_order_up_to' in cfg_src \
    or 'kit_equivalent_order_up_to' in (repo_root / 'supply_chain/supply_chain.py').read_text(), \
    'faithful flow mode missing'
print('contract check: tail wiring + faithful modes present — ok')


In [ ]:
# ---- build a smoke command from the knobs ----
def smoke_cmd(label, risk_level, seed, timesteps):
    cmd = [sys.executable, 'scripts/run_thesis_decision_ppo_smoke.py',
        '--label', label, '--output-root', str(output_root),
        '--algo', ALGO, '--action-space-mode', ACTION_SPACE_MODE,
        '--reward-mode', REWARD_MODE,
        '--ret-ladder-w-sc', str(LADDER_W_SC),
        '--ret-ladder-w-rc', str(LADDER_W_RC),
        '--ret-ladder-w-ef', str(LADDER_W_EF),
        '--risk-level', risk_level,
        '--risk-occurrence-mode', RISK_OCCURRENCE_MODE,
        '--raw-material-flow-mode', RAW_MATERIAL_FLOW_MODE,
        '--raw-material-order-up-to-multiplier', str(ORDER_UP_TO_MULTIPLIER),
        '--train-timesteps', str(timesteps), '--eval-episodes', str(EVAL_EPISODES),
        '--seed', str(seed), '--garrido-cfis', GARRIDO_CFIS,
        '--policy-net-arch', POLICY_NET_ARCH, '--learning-rate', str(LEARNING_RATE),
        '--n-envs', str(N_ENVS),
        '--no-eval-ai-on-garrido-cfis', '--include-static-grid', '--device', 'cpu']
    if STOCHASTIC_PT:
        cmd.append('--stochastic-pt')
    cmd.append('--norm-reward' if NORM_REWARD else '--no-norm-reward')
    if TRAIN_CFIS:
        cmd += ['--train-cfis', TRAIN_CFIS]
    if ALGO == 'dmlpa_ppo':
        cmd += ['--history-window', str(HISTORY_WINDOW)]
    return cmd


In [ ]:
# ---- run the risk x seed grid (faithful env) ----
run_labels = []
for risk_level in RISK_LEVELS:
    for seed in SEEDS:
        label = f'tail_{ALGO}_{risk_level}_seed{seed}'
        run_labels.append((label, risk_level, seed))
        print('\n' + '=' * 80 + f'\nRUN {label}\n' + '=' * 80, flush=True)
        try:
            run_cmd(smoke_cmd(label, risk_level, seed, TRAIN_TIMESTEPS), cwd=repo_root)
        except Exception as exc:
            print(f'RUN {label} FAILED: {exc}', flush=True)


In [ ]:
# ---- analysis: PPO vs BEST static on the CORRECT (tail) metrics ----
import pandas as pd, numpy as np

PRIMARY = ['ret_mean_all_orders_zero_unfulfilled_mean', 'flow_fill_rate_mean']
TAIL    = ['ret_p10_all_mean', 'ret_p50_all_mean', 'stockout_week_pct_mean',
           'pct_case_fill_rate_mean']
DECOMP  = ['re_fr_contribution_all_mean', 're_ap_contribution_all_mean',
           're_rp_contribution_all_mean', 're_dp_rp_contribution_all_mean']

def load(label):
    p = output_root / label / 'policy_summary.csv'
    return pd.read_csv(p) if p.exists() else None

rows = []
for label, risk_level, seed in run_labels:
    df = load(label)
    if df is None:
        print(f'[{label}] no policy_summary.csv (failed/timed out)'); continue
    df['policy'] = df['policy'].astype(str)
    ppo = df[df['policy'].str.contains('ppo')]
    grid = df[df['policy'].str.startswith('static_grid')]
    if ppo.empty or grid.empty:
        print(f'[{label}] missing ppo/grid rows'); continue
    ppo_r = ppo.iloc[0]
    # 'best static' = highest PRIMARY all-order ReT (the well-chosen static)
    pcol = PRIMARY[0]
    best = grid.loc[grid[pcol].astype(float).idxmax()]
    print('\n' + '#' * 80)
    print(f'# {label}  (best static by {pcol} = {best["policy"]})')
    print('#' * 80)
    print(f'{"metric":42s} {"PPO":>11s} {"best_static":>13s} {"delta":>11s}')
    for col in PRIMARY + TAIL + DECOMP:
        if col not in df.columns:
            continue
        pv, sv = float(ppo_r[col]), float(best[col])
        flag = ''
        if col in PRIMARY + ['ret_p10_all_mean', 'ret_p50_all_mean', 'flow_fill_rate_mean']:
            flag = '  <-- PPO better' if pv > sv + 1e-9 else ('  (ties)' if abs(pv-sv) <= 1e-9 else '')
        print(f'{col:42s} {pv:11.4f} {sv:13.4f} {pv - sv:+11.4f}{flag}')
        rows.append({'run': label, 'algo': ALGO, 'risk': risk_level, 'seed': seed,
                     'metric': col, 'ppo': pv, 'best_static': sv, 'delta': pv - sv})
    # also report best-static chosen ON the tail itself (not by mean) — strongest comparison
    if 'ret_p10_all_mean' in df.columns:
        bp10 = float(grid['ret_p10_all_mean'].astype(float).max())
        pp10 = float(ppo_r['ret_p10_all_mean'])
        print(f'\n  tail p10: PPO={pp10:.4f}  best-static-by-p10={bp10:.4f}  '
              f'-> PPO {"WINS" if pp10 > bp10 else "loses"} on the tail')

if rows:
    vdf = pd.DataFrame(rows)
    out = output_root / 'tail_resilience_summary.csv'
    vdf.to_csv(out, index=False); print('\nSaved', out)
    print('\n=== Delta (PPO - best_static), tail+primary, across runs ===')
    key = vdf[vdf['metric'].isin(PRIMARY + ['ret_p10_all_mean', 'ret_p50_all_mean'])]
    print(key.pivot_table(index='metric', columns='run', values='delta').to_string())
    print('\nReading: positive delta on ret_mean_all_orders / ret_p10 / flow_fill = RL separates;\n'
          'lower stockout_week_pct also favors RL. If all ~0 or negative, Track A is closed on\n'
          'resilience too, not just on mean service.')
else:
    print('No rows produced.')


In [ ]:
# ---- OPTIONAL convergence diagnostic: eval metric vs train budget ----
# Enable by setting CONVERGENCE_BUDGETS in the knobs cell. Helps David see whether the
# policy is LEARNING (metric climbs toward/past the static line) or STUCK (flat).
if CONVERGENCE_BUDGETS:
    import pandas as pd, numpy as np
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    risk_level, seed = RISK_LEVELS[0], SEEDS[0]
    metric = 'ret_mean_all_orders_zero_unfulfilled_mean'
    curve, static_ref = [], None
    for b in CONVERGENCE_BUDGETS:
        label = f'conv_{ALGO}_{risk_level}_seed{seed}_b{b}'
        print('\n' + '=' * 70 + f'\nCONV budget={b}\n' + '=' * 70, flush=True)
        try:
            run_cmd(smoke_cmd(label, risk_level, seed, b), cwd=repo_root)
        except Exception as exc:
            print(f'budget {b} FAILED: {exc}'); continue
        p = output_root / label / 'policy_summary.csv'
        if not p.exists():
            continue
        df = pd.read_csv(p); df['policy'] = df['policy'].astype(str)
        ppo = df[df['policy'].str.contains('ppo')]
        grid = df[df['policy'].str.startswith('static_grid')]
        if ppo.empty:
            continue
        curve.append((b, float(ppo.iloc[0][metric])))
        if static_ref is None and not grid.empty:
            static_ref = float(grid[metric].astype(float).max())
    if curve:
        xs, ys = zip(*curve)
        plt.figure(figsize=(7, 4))
        plt.plot(xs, ys, 'o-', label=f'{ALGO} (eval {metric})')
        if static_ref is not None:
            plt.axhline(static_ref, ls='--', color='gray', label='best static')
        plt.xlabel('train timesteps'); plt.ylabel(metric); plt.legend(); plt.grid(alpha=.3)
        plt.title(f'Convergence — {ALGO} @ {risk_level} seed{seed}')
        fig_path = output_root / 'convergence_curve.png'
        plt.savefig(fig_path, bbox_inches='tight', dpi=120); print('Saved', fig_path)
        for b, y in curve:
            print(f'  budget {b:>8d} -> {metric} = {y:.4f}')
else:
    print('CONVERGENCE_BUDGETS empty — skipping convergence diagnostic.')


In [ ]:
# ---- keep Kaggle output small: export only tail_resilience, not the cloned repo ----
if IN_KAGGLE:
    shutil.rmtree(repo_root, ignore_errors=True)
    print('cleaned cloned repo from Kaggle output:', repo_root)
